# Silver | Databricks (SCB 2023–2025; electricity 2025)

Run all cells in order after the Bronze job succeeds. This notebook reads the four SCB Bronze Delta tables and the electricity Bronze Delta table. It makes no API requests. All four validated Silver tables are overwritten on rerun.

Population comes from TAB638 for 2023–2024 and TAB5557 for 2025. The 2025 total uses SCB's new category codes. Electricity prices are hourly before October 2025 and every 15 minutes afterwards. Times in Silver are UTC; the source's malformed interval at the autumn clock change is corrected in Silver and its original end is retained.


In [0]:
import json
import math
import re
from collections import defaultdict
from datetime import date, datetime, time, timedelta, timezone
from itertools import product
from zoneinfo import ZoneInfo

from pyspark.sql.types import (
    DateType, DoubleType, IntegerType, LongType, StringType,
    StructField, StructType, TimestampType,
)

CATALOG = "db_labb2"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")
print("Reading", BRONZE, "→ writing", SILVER)


In [0]:
def dimension_codes(payload, dimension_id):
    """Return JSON-stat codes in the same order as the value array."""
    index = payload["dimension"][dimension_id]["category"]["index"]
    if isinstance(index, dict):
        return [code for code, _ in sorted(index.items(), key=lambda pair: pair[1])]
    if isinstance(index, list):
        return index
    raise ValueError(f"Unsupported category.index for {dimension_id}: {type(index)}")


def read_scb_rows(table_id, expected_dimensions, expected_codes):
    """Read twelve raw responses and expand their validated JSON-stat values."""
    table = f"{BRONZE}.scb_{table_id.lower()}"
    responses = spark.table(table).select(
        "source", "table_id", "batch_no", "payload_json"
    ).collect()
    batches = {row.batch_no for row in responses}
    if len(responses) != 12 or batches != set(range(1, 13)):
        raise ValueError(f"{table}: expected batches 1–12, found {sorted(batches)}")

    rows = []
    for response in responses:
        if response.source != "SCB" or response.table_id != table_id:
            raise ValueError(f"{table}: incorrect source/table_id")
        payload = json.loads(response.payload_json)
        dimensions = payload["id"]
        if set(dimensions) != set(expected_dimensions):
            raise ValueError(f"{table}: unexpected dimensions {dimensions}")
        codes = [dimension_codes(payload, dimension) for dimension in dimensions]
        if [len(items) for items in codes] != payload["size"]:
            raise ValueError(f"{table}: dimension size mismatch")
        for dimension, wanted in expected_codes.items():
            if set(codes[dimensions.index(dimension)]) != set(wanted):
                raise ValueError(f"{table}: unexpected {dimension} codes")
        values = payload["value"]
        if len(values) != math.prod(len(items) for items in codes):
            raise ValueError(f"{table}: value count mismatch")
        for combination, value in zip(product(*codes), values):
            rows.append({**dict(zip(dimensions, combination)), "value": value})
    print(table, "→", len(rows), "observations")
    return rows


def nonnegative_int(value, context):
    """Convert a nonnegative, integral SCB observation to an integer."""
    if isinstance(value, bool) or value is None or not isinstance(value, (int, float)):
        raise ValueError(f"{context}: missing or nonnumeric observation {value!r}")
    if not math.isfinite(value) or value < 0 or int(value) != value:
        raise ValueError(f"{context}: invalid count {value!r}")
    return int(value)


def check_keys(name, rows, indices, expected_count):
    """Verify row count, four-digit municipality codes, and unique business keys."""
    keys = [tuple(row[index] for index in indices) for row in rows]
    if len(rows) != expected_count or len(set(keys)) != expected_count:
        raise ValueError(f"{name}: expected {expected_count} distinct keys; got {len(rows)} rows")
    if any(re.fullmatch(r"[0-9]{4}", row[0]) is None for row in rows):
        raise ValueError(f"{name}: municipality codes must contain four digits")
    if len({row[0] for row in rows}) != 290:
        raise ValueError(f"{name}: expected 290 municipalities")
    print(name, "validated:", len(rows), "rows, 290 municipalities")


In [0]:
# The two population sources have different JSON-stat dimensions and measure codes.
pop_early = read_scb_rows(
    "TAB638", ["Region", "ContentsCode", "Tid"],
    {"ContentsCode": ["BE0101N1"], "Tid": ["2023", "2024"]},
)
pop_2025 = read_scb_rows(
    "TAB5557", ["Region", "Civilstand", "Alder", "Kon", "ContentsCode", "Tid"],
    {"Civilstand": ["SC"], "Alder": ["TotSA"], "Kon": ["TotSa"],
     "ContentsCode": ["000007ME"], "Tid": ["2025"]},
)
vehicle_source = read_scb_rows(
    "TAB1278", ["Region", "Fordonsslag", "ContentsCode", "Tid"],
    {"Fordonsslag": ["10"], "ContentsCode": ["TK1001AC"],
     "Tid": ["2023", "2024", "2025"]},
)
registration_source = read_scb_rows(
    "TAB3277", ["Region", "Drivmedel", "ContentsCode", "Tid"],
    {"Drivmedel": ["120", "140"], "ContentsCode": ["TK1001AA"],
     "Tid": [f"{year}M{month:02d}" for year in (2023, 2024, 2025)
             for month in range(1, 13)]},
)

population_rows = [
    (item["Region"], int(item["Tid"]),
     nonnegative_int(item["value"], "population"), item["ContentsCode"])
    for item in pop_early + pop_2025
]
vehicle_rows = [
    (item["Region"], int(item["Tid"]), item["Fordonsslag"],
     nonnegative_int(item["value"], "passenger cars"), item["ContentsCode"])
    for item in vehicle_source
]
registration_rows = [
    (item["Region"], date(int(item["Tid"][:4]), int(item["Tid"][5:]), 1),
     item["Drivmedel"], nonnegative_int(item["value"], "registrations"),
     item["ContentsCode"])
    for item in registration_source
]

check_keys("population", population_rows, (0, 1), 290 * 3)
check_keys("vehicles_in_use", vehicle_rows, (0, 1), 290 * 3)
check_keys("registrations", registration_rows, (0, 1, 2), 290 * 36 * 2)
if not ({row[0] for row in population_rows} == {row[0] for row in vehicle_rows}
        == {row[0] for row in registration_rows}):
    raise ValueError("Municipality coverage differs among SCB sources")
if {row[1] for row in population_rows} != {2023, 2024, 2025}:
    raise ValueError("Population year coverage is incomplete")
if any(row[2] != "10" for row in vehicle_rows):
    raise ValueError("Unexpected vehicle type")
if {row[2] for row in registration_rows} != {"120", "140"}:
    raise ValueError("Unexpected registration fuel code")


In [0]:
# Convert each Bronze day/area response to UTC intervals; retain the raw source end.
STOCKHOLM = ZoneInfo("Europe/Stockholm")
PRICE_START, PRICE_END = date(2025, 1, 1), date(2025, 12, 31)
AREAS = {"SE1", "SE2", "SE3", "SE4"}


def utc_timestamp(value):
    """Parse a source timestamp containing a UTC offset and normalize it to UTC."""
    parsed = datetime.fromisoformat(value)
    if parsed.tzinfo is None or parsed.utcoffset() is None:
        raise ValueError(f"Electricity timestamp has no UTC offset: {value}")
    return parsed.astimezone(timezone.utc)


price_raw = spark.table(f"{BRONZE}.electricity_price_raw").select(
    "source", "price_date", "price_area", "payload_json"
).collect()
daily = {}
price_rows = []
corrections = []
for response in price_raw:
    day, area = response.price_date, response.price_area
    if (response.source != "Elpriset just nu" or area not in AREAS
            or day < PRICE_START or day > PRICE_END or (day, area) in daily):
        raise ValueError(f"Invalid/duplicate electricity response: {day}, {area}, {response.source}")
    payload = json.loads(response.payload_json)
    if not isinstance(payload, list) or not payload:
        raise ValueError(f"Missing electricity intervals for {day} {area}")
    expected_minutes = 60 if day < date(2025, 10, 1) else 15
    intervals = []
    for item in payload:
        start = utc_timestamp(item["time_start"])
        raw_end = utc_timestamp(item["time_end"])
        end = start + timedelta(minutes=expected_minutes)
        if start.astimezone(STOCKHOLM).date() != day:
            raise ValueError(f"Interval belongs to another local date: {day} {area} {start}")
        if raw_end != end:
            # One known source error per area: 02:45+02:00 → 03:00+01:00 (75 minutes).
            if not (day == date(2025, 10, 26)
                    and start == datetime(2025, 10, 26, 0, 45, tzinfo=timezone.utc)
                    and raw_end == datetime(2025, 10, 26, 2, 0, tzinfo=timezone.utc)
                    and end == datetime(2025, 10, 26, 1, 0, tzinfo=timezone.utc)):
                raise ValueError(f"Unexpected electricity interval: {day} {area} {item}")
            corrections.append((day, area))
        amounts = (item["SEK_per_kWh"], item["EUR_per_kWh"], item["EXR"])
        if any(isinstance(v, bool) or not isinstance(v, (int, float))
               or not math.isfinite(v) for v in amounts):
            raise ValueError(f"Invalid electricity price/rate: {day} {area} {item}")
        intervals.append((start, end))
        price_rows.append((area, day, start, end, float(amounts[0]),
                           float(amounts[1]), float(amounts[2]),
                           float(expected_minutes), raw_end))
    daily[(day, area)] = sorted(intervals)

expected_days = {(PRICE_START + timedelta(days=n), area)
                 for n in range((PRICE_END - PRICE_START).days + 1)
                 for area in AREAS}
if set(daily) != expected_days:
    raise ValueError(f"Electricity day/area coverage: missing {len(expected_days - set(daily))}, "
                     f"unexpected {len(set(daily) - expected_days)}")
if len(corrections) != 4 or {area for _, area in corrections} != AREAS:
    raise ValueError(f"Expected four clock-change source corrections: {corrections}")
for (day, area), intervals in daily.items():
    expected_first = datetime.combine(day, time.min, STOCKHOLM).astimezone(timezone.utc)
    expected_last = datetime.combine(day + timedelta(days=1), time.min,
                                     STOCKHOLM).astimezone(timezone.utc)
    minutes = 60 if day < date(2025, 10, 1) else 15
    if (intervals[0][0] != expected_first or intervals[-1][1] != expected_last
            or len(intervals) != (expected_last - expected_first).total_seconds() / (minutes * 60)
            or any(left[1] != right[0] for left, right in zip(intervals, intervals[1:]))):
        raise ValueError(f"Electricity intervals have a gap/overlap: {day} {area}")
if len(price_rows) != 61548 or len({(row[0], row[2]) for row in price_rows}) != 61548:
    raise ValueError(f"Electricity expected 61,548 distinct intervals, got {len(price_rows)}")
print("electricity_prices validated:", len(price_rows), "intervals,",
      len(daily), "dates × areas,", len(corrections), "source end corrections")


In [0]:
# Write after every source and business key has passed validation.
population_schema = StructType([
    StructField("kommun_code", StringType(), False),
    StructField("year", IntegerType(), False),
    StructField("population", LongType(), False),
    StructField("ContentsCode", StringType(), False),
])
vehicle_schema = StructType([
    StructField("kommun_code", StringType(), False),
    StructField("year", IntegerType(), False),
    StructField("vehicle_type_code", StringType(), False),
    StructField("passenger_cars_in_use", LongType(), False),
    StructField("ContentsCode", StringType(), False),
])
registration_schema = StructType([
    StructField("kommun_code", StringType(), False),
    StructField("month", DateType(), False),
    StructField("fuel_code", StringType(), False),
    StructField("new_registrations", LongType(), False),
    StructField("ContentsCode", StringType(), False),
])
price_schema = StructType([
    StructField("price_area", StringType(), False),
    StructField("price_date", DateType(), False),
    StructField("time_start", TimestampType(), False),
    StructField("time_end", TimestampType(), False),
    StructField("sek_per_kwh", DoubleType(), False),
    StructField("eur_per_kwh", DoubleType(), False),
    StructField("exchange_rate", DoubleType(), False),
    StructField("duration_minutes", DoubleType(), False),
    StructField("source_time_end", TimestampType(), False),
])

for name, rows, schema in [
    ("population", population_rows, population_schema),
    ("vehicles_in_use", vehicle_rows, vehicle_schema),
    ("registrations", registration_rows, registration_schema),
    ("electricity_prices", price_rows, price_schema),
]:
    target = f"{SILVER}.{name}"
    (spark.createDataFrame(rows, schema)
          .write.format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable(target))
    written = spark.table(target).count()
    if written != len(rows):
        raise ValueError(f"{target}: wrote {written} of {len(rows)} rows")
    print(target, "→", written, "rows")
